### HW3: Wide data and linear models

##### Name: Ananya Agrawal
##### Andrew ID: ananyaa2

##### 38615: Computational Modelling, Statistical Analysis and Machine Learning in Science - Homework 3


#### Importing Libraries

In [57]:
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifier, LassoCV, Lasso
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#### Reading the Training Set Features, Training Set Labels and Test Set Features

In [34]:
TEST_X = "f24-38615-hw3/test_X.csv"
TRAIN_X = "f24-38615-hw3/train_X.csv"
TRAIN_Y = "f24-38615-hw3/train_y.csv"

In [58]:
x = pd.read_csv(TRAIN_X).drop(columns=['Unnamed: 0']) # Dropping the first column - ID 
x

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,ENSG00000001460,...,ENSG00000282651,ENSG00000282815,ENSG00000282939,ENSG00000283063,ENSG00000283439,ENSG00000283463,ENSG00000283526,ENSG00000283586,ENSG00000283632,ENSG00000283697
0,7.062725,0.026623,6.720413,5.449267,3.868619,4.587771,7.165112,4.643161,6.771731,4.750296,...,0.325987,-5.545564,-5.545564,-5.545564,-5.545564,4.014351,4.841392,-5.545564,5.855893,3.618253
1,5.965392,-5.431256,6.358498,4.161479,4.585293,4.326924,6.849703,4.391534,5.819945,3.435322,...,5.910874,-0.945029,3.750430,1.611211,-0.498573,3.430928,3.160435,-5.431256,4.413930,3.353496
2,7.892221,-5.851870,8.132992,5.986320,5.422599,4.728815,8.168477,6.289562,7.331591,5.336794,...,10.103565,-5.851870,6.498217,5.481945,-5.851870,5.137298,4.296777,-5.851870,5.345372,5.028567
3,6.826546,0.964851,5.998280,4.991435,4.963000,4.977695,7.149421,4.570863,6.008286,5.474553,...,2.442099,-5.994056,2.862038,1.909955,0.568120,4.768694,3.983207,-5.994056,4.609411,4.329472
4,7.059095,2.429954,6.746639,5.591316,5.111120,5.972938,7.576201,6.032083,6.470761,5.380887,...,5.553223,-5.870484,3.044916,-5.870484,0.018320,4.640575,4.954957,-5.870484,4.620774,4.464277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,7.009917,0.088022,6.814481,4.685804,3.039239,4.373218,6.826167,5.101784,6.327076,3.517293,...,5.244168,-5.619018,3.953899,-5.619018,-0.021027,4.055201,3.052880,-5.619018,4.560183,3.889471
440,4.772186,-5.062627,7.064273,5.505659,5.769194,4.307329,6.576986,4.018168,5.453594,3.499849,...,6.641965,-0.284152,4.238647,5.431223,0.785492,3.489883,-5.062627,-5.062627,4.467892,3.335732
441,7.741897,1.467622,6.547112,4.994184,5.273657,6.069989,7.170169,5.740295,6.115446,6.510703,...,5.963416,-5.380528,4.411300,2.265077,-5.380528,4.727461,4.502798,-5.380528,4.102909,4.326804
442,5.774051,-6.793718,7.218590,5.477760,5.042350,3.700158,7.349451,4.573089,6.003810,4.067563,...,6.036861,-6.793718,3.605128,2.503595,-6.793718,3.401223,4.962133,-6.793718,3.864213,3.200419


In [59]:
y = pd.read_csv(TRAIN_Y).drop(columns=['Unnamed: 0']) # Dropping the first column - ID 
y

,xml_neoplasm_histologic_grade
0,0
1,1
2,0
3,0
4,1
...,...
439,1
440,1
441,1
442,0


#### Exploring the data

In [38]:
x.describe()

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,ENSG00000001460,...,ENSG00000282651,ENSG00000282815,ENSG00000282939,ENSG00000283063,ENSG00000283439,ENSG00000283463,ENSG00000283526,ENSG00000283586,ENSG00000283632,ENSG00000283697
count,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,...,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000,444.000000
mean,6.798864,-1.251875,6.867482,5.092604,4.249074,4.951573,7.179084,5.241762,6.152933,4.498626,...,3.650685,-4.019460,2.852973,0.532640,-3.685311,4.131878,3.000574,-5.558757,4.313466,3.944751
std,0.657958,3.487385,0.649468,0.557108,0.974955,1.245837,0.635931,0.803235,0.556726,0.770503,...,3.537927,2.779953,2.558302,3.758160,2.842395,0.743549,2.396337,1.893713,0.798173,0.755219
min,3.865658,-8.167826,3.921938,2.686933,0.635382,1.559740,4.091881,1.961100,3.491674,0.877428,...,-8.025671,-8.093499,-7.791345,-7.821501,-8.093499,1.289553,-7.174879,-8.167826,-0.430043,0.899681
25%,6.447793,-5.483300,6.546906,4.765809,3.630805,4.075300,6.844709,4.783474,5.828785,4.045615,...,2.240973,-5.966191,2.295934,-0.131155,-5.899284,3.706238,1.768129,-6.599244,3.882128,3.511135
50%,6.852772,0.213298,6.938184,5.093077,4.238058,4.896219,7.207127,5.155404,6.157952,4.492328,...,4.101297,-5.425883,3.333111,1.984213,-5.168411,4.119857,3.473621,-5.861161,4.348660,3.944781
75%,7.226415,1.184977,7.269660,5.435893,4.935047,5.948701,7.594792,5.664817,6.468815,4.950106,...,5.794548,-1.497067,4.331129,3.146919,-0.619910,4.611680,4.724469,-5.445569,4.810073,4.425701
max,8.642886,9.451403,8.598549,6.988089,7.650225,8.220311,8.916491,8.810685,7.702358,7.233636,...,12.370689,5.137566,7.188701,7.083282,2.026253,6.738504,7.181350,1.982121,6.661020,6.684243


In [39]:
y.describe()

,xml_neoplasm_histologic_grade
count,444.000000
mean,0.572072
std,0.495336
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [44]:
print(x.info())
print(y.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444 entries, 0 to 443
Columns: 17970 entries, ENSG00000000003 to ENSG00000283697
dtypes: float64(17970)
memory usage: 60.9 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444 entries, 0 to 443
Data columns (total 1 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   xml_neoplasm_histologic_grade  444 non-null    int64
dtypes: int64(1)
memory usage: 3.6 KB
None


In [43]:
print(x.dtypes)
print(y.dtypes)

ENSG00000000003    float64
ENSG00000000005    float64
ENSG00000000419    float64
ENSG00000000457    float64
ENSG00000000938    float64
                    ...   
ENSG00000283463    float64
ENSG00000283526    float64
ENSG00000283586    float64
ENSG00000283632    float64
ENSG00000283697    float64
Length: 17970, dtype: object
xml_neoplasm_histologic_grade    int64
dtype: object


In [41]:
check_null = x.isnull().sum()
null_rows_with_data = check_null[check_null > 0]  # Filter only rows with null values
print(null_rows_with_data)
check_null = y.isnull().sum()
null_rows_with_data = check_null[check_null > 0]  # Filter only rows with null values
print(null_rows_with_data)

Series([], dtype: int64)
Series([], dtype: int64)


#### Finding Outliers

In [48]:
total_outliers = 0
dataset_filtered = x

for column in x:
  Q1 = x[column].quantile(0.25)
  Q3 = x[column].quantile(0.75)
  IQR = Q3 - Q1

  # extending the range of IQR for identifying outliers in the given dataset from 1.5 times to 6 times
  lower_bound = Q1 - 6 * IQR
  upper_bound = Q3 + 6 * IQR

  outliers = x[(x[column] < lower_bound) | (x[column] > upper_bound)]
  total_outliers += len(outliers[column])
  dataset_filtered = dataset_filtered[(dataset_filtered[column] >= lower_bound) & (dataset_filtered[column] <= upper_bound)]

avg_outliers = (total_outliers / len(x))
print("Average number of Outliers detected per column ", avg_outliers)

Average number of Outliers detected per column  8.457207207207206


#### Performing Binary Classification (0/1) using Linear Models

In [60]:
test_x = pd.read_csv(TEST_X)
tt_x = test_x.drop(columns=['Unnamed: 0'])
tt_x

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,ENSG00000001460,...,ENSG00000282651,ENSG00000282815,ENSG00000282939,ENSG00000283063,ENSG00000283439,ENSG00000283463,ENSG00000283526,ENSG00000283586,ENSG00000283632,ENSG00000283697
0,7.503849,1.685440,8.090089,5.546588,3.412025,5.871539,8.395029,5.894996,6.313896,4.286261,...,3.626396,-2.640463,-7.278265,0.721287,-7.278265,4.672536,5.153675,-7.278265,4.843800,4.302864
1,6.112919,-5.425877,5.604743,4.248246,3.374033,3.370075,6.804015,4.646287,6.236134,4.167541,...,-5.425877,-2.143410,1.956699,1.377768,-5.425877,4.560829,3.735085,-5.425877,4.105789,4.803357
2,6.183846,1.217355,6.093903,4.403216,5.722867,6.123209,7.680258,4.983386,5.626569,3.518791,...,11.317818,-5.259782,5.093316,3.800607,-5.259782,4.966710,2.829487,-5.259782,5.118704,5.024979
3,6.325535,-0.197432,6.722632,4.509093,4.941256,3.700171,7.492606,5.031053,5.859242,4.252114,...,3.536141,-1.362703,3.480703,0.776383,-0.999629,4.794226,-5.399485,-5.399485,4.660878,4.684343
4,7.162383,-5.539710,6.186110,5.270282,5.412103,3.781568,7.285779,4.847552,6.337205,5.622222,...,5.112772,-5.539710,4.369268,2.927849,0.791397,4.542333,4.628775,-5.539710,4.812787,4.524834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,6.949029,2.192625,7.613706,5.131517,4.292879,5.337764,7.668408,5.159289,5.645662,3.082916,...,5.515318,-5.785898,4.229114,2.875184,-0.303496,3.322246,3.683199,-5.785898,4.730559,3.251810
106,7.419322,0.472349,6.723539,5.176782,4.297442,4.591905,7.171547,6.693665,5.782247,5.965700,...,2.737453,-1.781791,3.509502,0.021815,-6.758708,4.684549,4.795272,-6.758708,4.234056,4.614238
107,7.135800,0.479097,7.051451,6.042828,4.523706,7.250545,7.577726,5.622388,6.447874,4.835028,...,5.951147,-5.960456,4.025253,2.785925,-5.960456,4.236003,5.597498,-5.960456,5.279498,3.813894
108,6.859635,2.159767,7.254992,4.900402,4.256251,6.503227,6.810730,6.036136,6.282137,4.745579,...,4.462337,-5.553034,4.325258,2.593648,-5.553034,4.727639,2.380364,-5.553034,5.523681,4.088765


In [61]:
# Taking X and Y for the data arrays
# Splitting the training dataset into training and validation sets considering 70% is training and 30% is validation
X_train, X_val, Y_train, Y_val = train_test_split(x, y, test_size=0.3, random_state=101) # maintaining a random state for reproducibility

In [54]:
# Initialize models to be tested in a pipeline
models = {
    'Linear Regression' : LinearRegression(),
    'Logistic Regression (L1 regularization)': LogisticRegression(penalty='l1', solver='saga', max_iter=50000),
    'Logistic Regression (L2 regularization)': LogisticRegression(penalty='l2', solver='saga', max_iter=50000),
    'Logistic Regression (ElasticNet (L1 + L2 regularization))':LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=50000),
    'Ridge Classifier': RidgeClassifier(),
    'Lasso (as Logistic)': Lasso(max_iter=1000),
    'Support Vector Classifier': SVC(kernel='linear')
}

# # Define a function to evaluate models
# def evaluate_model(model, X_train, X_val, Y_train, Y_val):
#     # Fit the model
#     model.fit(X_train, Y_train)
    
#     # Predict on validation set
#     y_pred = model.predict(X_val)
    
#     # Compute accuracy and F1-score
#     accuracy = accuracy_score(Y_val, y_pred)
#     f1 = f1_score(Y_val, y_pred)
    
#     return accuracy, f1

In [55]:
# Evaluate each model
results = {}
for model_name, model in models.items():
    accuracy, f1 = evaluate_model(model, X_train, Y_train, X_val, Y_val)
    results[model_name] = {'Accuracy': accuracy, 'F1-score': f1}

results

ValueError: Found input variables with inconsistent numbers of samples: [310, 134]

In [ ]:
# Standardize data
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled = scaler.transform(X_val)

In [ ]:
def evaluate_model(model, X_train, tt_x, Y_train):
    # Fit the model
    model.fit(X_train, Y_train)
    
    # Predict on validation set
    y_pred = model.predict(tt_x)
    
    # # Compute accuracy and F1-score
    # accuracy = accuracy_score(Y_val, y_pred)
    # f1 = f1_score(Y_val, y_pred)
    
    # return accuracy, f1

# results = {}
# for model_name, model in models.items():
#     accuracy, f1 = evaluate_model(model, X_train, Y_train, X_val, Y_val)
#     results[model_name] = {'Accuracy': accuracy, 'F1-score': f1}

# results

In [62]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_val)

logreg_l1 = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=50000, C=0.7, l1_ratio=0.7)
logreg_l1.fit(x_train_scaled, Y_train)
y_pred_l1_scaled = logreg_l1.predict(x_test_scaled)
print("L1 Logistic Regression Accuracy with Scaling: ", accuracy_score(Y_val, y_pred_l1_scaled))

/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


L1 Logistic Regression Accuracy with Scaling:  0.7388059701492538


In [63]:
y_pred_gb = logreg_l1.predict(tt_x)
y_pred_gb

/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


array([1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [64]:
preds = pd.DataFrame(y_pred_gb)
op = pd.concat([test_x['Unnamed: 0'], preds], axis=1)
op.to_csv('testing1.csv', index=False)

In [ ]:
svc = SVC(kernel='linear')
svc.fit(X_train, Y_train)
y_pred_svc = svc.predict(X_val)
print("Linear SVC Accuracy: ", accuracy_score(Y_val, y_pred_svc))

In [ ]:
# Lasso (L1 regularization)
logreg_l1 = LogisticRegression(penalty='l1', solver='saga', max_iter=10000)
logreg_l1.fit(X_train, y_train)
y_pred_l1 = logreg_l1.predict(X_test)
print("L1 Logistic Regression Accuracy: ", accuracy_score(y_test, y_pred_l1))

In [ ]:
# Ridge (L2 regularization)
logreg_l2 = LogisticRegression(penalty='l2', solver='saga', max_iter=10000)
logreg_l2.fit(X_train, y_train)
y_pred_l2 = logreg_l2.predict(X_test)
print("L2 Logistic Regression Accuracy: ", accuracy_score(y_test, y_pred_l2))

input in pipeline

In [ ]:
# ElasticNet (L1 + L2 regularization)
logreg_elastic = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=10000)
logreg_elastic.fit(X_train, y_train)
y_pred_elastic = logreg_elastic.predict(X_test)
print("ElasticNet Logistic Regression Accuracy: ", accuracy_score(y_test, y_pred_elastic))

input in pipeline

add select kbest, add randomization/random guessing, complete 4, 5, 6 tasks and bonus Q

In [ ]:
# Step 2: Apply LASSO for feature selection
lasso = LassoCV(cv=10, random_state=42, max_iter=10000000)
lasso.fit(X_train, Y_train)

In [ ]:
sorte = np.argsort(np.abs(lasso.coef_))
def accuracy_check(idx) :
# Get indices of non-zero coefficients
    important_features_idx = sorte[idx:idx + 1]

    X_train_reduced = X_train.iloc[:, important_features_idx]
    X_test_reduced = X_test.iloc[:, important_features_idx]

    # Step 4: Logistic Regression on selected features
    log_reg = LogisticRegression(penalty='l2', max_iter=1000000)
    log_reg.fit(X_train_reduced, Y_train)

    # Step 5: Predict and check accuracy
    Y_pred = log_reg.predict(X_test_reduced)
    accuracy = accuracy_score(Y_test, Y_pred)

    print(f"Model accuracy: {accuracy:.4f}")
    return accuracy

In [ ]:
ref = {}
for i in range(0, len(sorte)):
    ref[i] = accuracy_check(i)

In [ ]:
hmm = sorted(ref.items(), key=lambda x:x[1])
hmm.reverse()

In [ ]:
hmm[0:5]

In [ ]:
def check_reduced_accuracy(idx, op = False):
    X_train_reduced = X_train.iloc[:, idx]
    X_test_reduced = X_test.iloc[:, idx]
    log_reg = LogisticRegression(max_iter=1000000, penalty='l2')
    log_reg.fit(X_train_reduced, Y_train)
    Y_pred = log_reg.predict(X_test_reduced)
    accuracy = accuracy_score(Y_test, Y_pred)

    if op:
        test_op_x = tt_x.iloc[:, idx]
        test_op_pred = log_reg.predict(test_op_x)
        print(test_op_pred)
        preds = pd.DataFrame(test_op_pred)
        plsgivegoodresult = pd.concat([tt_x_or['Unnamed: 0'], preds], axis=1)
        plsgivegoodresult.to_csv('plsgivegoodresult.csv', index=False)
        print("OP")
    return accuracy

In [ ]:
feats = [sorte[hmm[0][0]]]
cur_accuracy = check_reduced_accuracy(feats)
cur_accuracy

In [ ]:
# feats = [sorte[hmm[0][0]]]
# cur_accuracy = check_reduced_accuracy(feats)

# i = 1
# while len(feats) < 100:
#     copy = list(feats)
#     copy.append(sorte[hmm[i][0]])
#     temp_acc = check_reduced_accuracy(copy)
#     if (temp_acc > cur_accuracy):
#         cur_accuracy = temp_acc
#         feats = list(copy)
#         print(cur_accuracy, feats)
#     i += 1
#     if (i % 500 == 0):
#         print("i",i)


feats = [sorte[hmm[0][0]]]
cur_accuracy = check_reduced_accuracy(feats)

i = 1
while len(feats) < 8000:
    copy = list(feats)
    copy.append(sorte[hmm[i][0]])
    temp_acc = check_reduced_accuracy(copy)
    if (temp_acc >= 0.865 or temp_acc > cur_accuracy):
        cur_accuracy = temp_acc
        feats = list(copy)
        
    i += 1
    if (i % 500 == 0):
        print("i",i)
        print(cur_accuracy, feats)

In [ ]:
len(feats)
check_reduced_accuracy(feats, op=True)

In [9]:
# Separate features and labels

X_df = data_X.drop(columns=['Unnamed: 0'])
y_df = data_y['xml_neoplasm_histologic_grade']

features = list(X_df)
label = y_df

X, y = X_df[features].values, y_df[label].values

for n in range(0,4):
    print(f"Patient {str(n+1)}, \n  Features: {list(X[n])} \n  Label: {y[n]}")

Patient 1, 
  Features: [np.float64(7.0627255), np.float64(0.026622772), np.float64(6.720413), np.float64(5.4492674), np.float64(3.8686187), np.float64(4.5877705), np.float64(7.1651115), np.float64(4.6431613), np.float64(6.771731), np.float64(4.7502956), np.float64(5.6671405), np.float64(5.80309), np.float64(6.059881), np.float64(7.4579134), np.float64(5.32329), np.float64(5.8666224), np.float64(5.932765), np.float64(5.1662254), np.float64(4.892952), np.float64(0.19447088), np.float64(7.3821573), np.float64(6.409154), np.float64(7.739046), np.float64(4.885576), np.float64(4.626525), np.float64(0.9983146), np.float64(1.6215897), np.float64(5.340644), np.float64(7.770936), np.float64(5.7776365), np.float64(5.059549), np.float64(7.1987247), np.float64(3.3983395), np.float64(2.4545858), np.float64(6.5345554), np.float64(6.5627155), np.float64(4.818756), np.float64(5.61048), np.float64(5.4929953), np.float64(6.796422), np.float64(5.2059183), np.float64(6.1969914), np.float64(2.8795063), np.

In [17]:
# Split data 70%-30% into training set and test set
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print (f'X_train: {X_train.shape} \n X_val: {X_val.shape} \n y_train: {y_train.shape} \n y_val: {y_val.shape}')

X_train: (310, 17970) 
 X_val: (134, 17970) 
 y_train: (310,) 
 y_val: (134,)


In [18]:
# Train the model
from sklearn.linear_model import LogisticRegression

# Set regularization rate
reg = 0.01

# train a logistic regression model on the training set
model = LogisticRegression(C=1/reg, solver="liblinear").fit(X_train, y_train)
print (model)

LogisticRegression(C=100.0, solver='liblinear')


In [19]:
predictions = model.predict(X_val)
print('Predicted labels: ', predictions)
print('Actual labels:    ' ,y_val)

Predicted labels:  [1 1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 0 1 1 1 0 1
 1 0 1 0 1 1 0 1 0 1 1 1 1 0 1 1 0 0 0 0 1 1 1 0 0 0 1 1 1 1 0 1 1 0 0 1 1
 1 1 1 0 1 0 0 1 1 1 1 1 1 1 0 0 0 1 0 0 1 0 1 1 1 1 0 1 0 0 0 0 1 0 1 0 1
 1 0 1 1 1 1 0 0 1 0 1 1 1 0 1 0 0 1 1 0 0 0 0]
Actual labels:     [1 1 0 0 1 0 1 0 1 1 0 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 0 1 1 1
 1 0 1 0 1 0 0 1 0 0 0 1 1 0 1 1 0 0 0 1 1 1 0 0 0 0 1 1 1 1 0 1 1 1 0 0 1
 1 0 1 0 1 0 0 1 1 1 1 1 0 1 0 0 0 1 0 0 1 0 1 1 1 1 0 1 0 0 0 0 0 0 1 0 1
 1 1 1 1 0 1 0 0 1 0 1 1 1 0 1 1 0 1 1 0 1 0 0]


In [20]:
from sklearn.metrics import accuracy_score

print(f"Accuracy score: {accuracy_score(y_val, predictions)}")

Accuracy score: 0.835820895522388


In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_val, predictions))

              precision    recall  f1-score   support

           0       0.84      0.75      0.80        57
           1       0.83      0.90      0.86        77

    accuracy                           0.84       134
   macro avg       0.84      0.83      0.83       134
weighted avg       0.84      0.84      0.83       134



In [22]:
# Train the model
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
import numpy as np

# Define preprocessing for numeric columns (normalize them so they're on the same scale)
numeric_features = [0,1,2,3,4,5,6]
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])
# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)])
# Create preprocessing and training pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('logregressor', LogisticRegression(C=1/reg, solver="liblinear"))])
# fit the pipeline to train a logistic regression model on the training set
model = pipeline.fit(X_train, (y_train))
print (model)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  [0, 1, 2, 3, 4, 5, 6])])),
                ('logregressor',
                 LogisticRegression(C=100.0, solver='liblinear'))])


In [23]:
# Get predictions from test data
predictions = model.predict(X_val)
y_scores = model.predict_proba(X_val)

In [24]:
print('Accuracy:', accuracy_score(y_val, predictions))


Accuracy: 0.664179104477612


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Lasso
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

# Prepare the data (drop the patient ID columns)
# X = data_X.drop(columns=['Unnamed: 0'])
# y = data_y['xml_neoplasm_histologic_grade']

# Split the data into training and validation sets (70/30)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# Initialize models to be tested
# models = {
#     'Logistic Regression': LogisticRegression(max_iter=1000),
#     'Ridge Classifier': RidgeClassifier(),
#     'Lasso (as Logistic)': Lasso(max_iter=1000)
# }

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Ridge Classifier': RidgeClassifier()
}

# Re-run the evaluation for the two models
results = {}
for model_name, model in models.items():
    accuracy, f1 = evaluate_model(model, X_train_scaled, y_train, X_val_scaled, y_val)
    results[model_name] = {'Accuracy': accuracy, 'F1-score': f1}

results

# Define a function to evaluate models
def evaluate_model(model, X_train, y_train, X_val, y_val):
    # Fit the model
    model.fit(X_train, y_train)
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    
    # Compute accuracy and F1-score
    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    return accuracy, f1

# Standardize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Evaluate each model
results = {}
for model_name, model in models.items():
    accuracy, f1 = evaluate_model(model, X_train_scaled, y_train, X_val_scaled, y_val)
    results[model_name] = {'Accuracy': accuracy, 'F1-score': f1}

results


{'Logistic Regression': {'Accuracy': 0.8202247191011236,
  'F1-score': np.float64(0.8431372549019608)},
 'Ridge Classifier': {'Accuracy': 0.8314606741573034,
  'F1-score': np.float64(0.8571428571428571)}}

In [28]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression with L1 regularization
log_l1 = LogisticRegression(penalty='l1', solver='saga', max_iter=1000)
log_l1.fit(X_train_scaled, y_train)

# Predict on validation set
y_pred = log_l1.predict(X_val_scaled)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Accuracy: {accuracy}")
print(f"F1-score: {f1}")


Accuracy: 0.8314606741573034
F1-score: 0.8484848484848485


/opt/homebrew/anaconda3/envs/CMU_ML/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define the hyperparameter grid for regularization strength
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

# Initialize the logistic regression with L2 regularization
log_reg = LogisticRegression(max_iter=1000)

# Perform Grid Search with cross-validation
grid_search = GridSearchCV(log_reg, param_grid, scoring='f1', cv=5)
grid_search.fit(X_train_scaled, y_train)

# Best parameters and model
best_log_reg = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Params: {best_params}")
print(f"Best F1-Score from CV: {best_score}")


Best Params: {'C': 0.01}
Best F1-Score from CV: 0.8611048863990041


In [30]:
from sklearn.linear_model import RidgeClassifier

# Define the hyperparameter grid for alpha
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Initialize Ridge Classifier
ridge = RidgeClassifier()

# Perform Grid Search with cross-validation
grid_search_ridge = GridSearchCV(ridge, param_grid, scoring='f1', cv=5)
grid_search_ridge.fit(X_train_scaled, y_train)

# Best parameters and model
best_ridge = grid_search_ridge.best_estimator_
best_ridge_params = grid_search_ridge.best_params_
best_ridge_score = grid_search_ridge.best_score_

print(f"Best Params for Ridge: {best_ridge_params}")
print(f"Best F1-Score from CV for Ridge: {best_ridge_score}")


Best Params for Ridge: {'alpha': 0.01}
Best F1-Score from CV for Ridge: 0.8605634254599984


In [31]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

# Lasso for feature selection
lasso = Lasso(alpha=0.01, max_iter=1000).fit(X_train_scaled, y_train)

# Select features that are non-zero in Lasso
model = SelectFromModel(lasso, prefit=True)
X_train_selected = model.transform(X_train_scaled)
X_val_selected = model.transform(X_val_scaled)

# Use selected features for Logistic Regression or Ridge
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_selected, y_train)
y_pred_selected = log_reg.predict(X_val_selected)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred_selected)
f1 = f1_score(y_val, y_pred_selected)
print(f"Accuracy after Feature Selection: {accuracy}")
print(f"F1-score after Feature Selection: {f1}")


Accuracy after Feature Selection: 0.797752808988764
F1-score after Feature Selection: 0.82


In [32]:
from sklearn.preprocessing import PolynomialFeatures

# Add polynomial features of degree 2
poly = PolynomialFeatures(degree=2, interaction_only=True)
X_train_poly = poly.fit_transform(X_train_scaled)
X_val_poly = poly.transform(X_val_scaled)

# Re-train Logistic Regression or Ridge with polynomial features
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_poly, y_train)
y_pred_poly = log_reg.predict(X_val_poly)

# Evaluate the model
accuracy_poly = accuracy_score(y_val, y_pred_poly)
f1_poly = f1_score(y_val, y_pred_poly)
print(f"Accuracy with Polynomial Features: {accuracy_poly}")
print(f"F1-score with Polynomial Features: {f1_poly}")
